# Klasifikasi Sentimen 6000 Komentar YouTube terhadap Kontroversi LCC 4 Pilar Kalimantan Barat Menggunakan Fine-Tuning IndoBERTweet

Notebook versi terbaru ini dibuat agar lebih aman dijalankan di Google Colab.

## Perbaikan versi ini

Versi ini **tidak menggunakan scikit-learn** supaya tidak terkena error konflik `numpy`, `scipy`, dan `sklearn` yang sebelumnya terjadi di Colab.

Semua proses penting tetap ada:

1. Scraping komentar YouTube.
2. Preprocessing teks Bahasa Indonesia.
3. Pelabelan awal otomatis 3 kelas.
4. EDA lengkap dan gambar hasil.
5. Split data train-test tanpa scikit-learn.
6. Fine-tuning IndoBERTweet.
7. Evaluasi model tanpa scikit-learn.
8. Confusion matrix.
9. Simpan model.
10. Export dataset, hasil, dan ZIP project.

## Judul

**Klasifikasi Sentimen 6000 Komentar YouTube terhadap Kontroversi LCC 4 Pilar Kalimantan Barat Menggunakan Fine-Tuning IndoBERTweet**

## Label

| Label | Makna |
|---|---|
| `kritik_juri_panitia` | Komentar mengkritik juri, panitia, MPR, sistem penilaian, aturan, keputusan lomba, atau meminta evaluasi |
| `pro_peserta` | Komentar membela, mendukung, atau menunjukkan simpati kepada peserta/siswa/sekolah |
| `netral` | Komentar bertanya, meminta link, memberi info, spam, atau tidak menunjukkan sikap jelas |

> Jika satu video tidak cukup 6000 komentar, tambahkan beberapa URL video relevan pada variabel `YOUTUBE_URLS`.


## 1. Instalasi Library Aman untuk Colab

In [ ]:
# ============================================================
# INSTALL LIBRARY AMAN
# Jalankan cell ini sekali saja. Jika diminta restart runtime, restart lalu lanjutkan dari cell import.
# Versi ini sengaja TIDAK memakai scikit-learn agar aman dari konflik sklearn/scipy/numpy.
# ============================================================

!pip -q install --no-cache-dir     google-api-python-client     pandas==2.2.2     numpy==1.26.4     matplotlib==3.8.4     openpyxl==3.1.5     Sastrawi     transformers     datasets     accelerate     joblib

print("Instalasi selesai. Kalau muncul peringatan restart runtime, lakukan Runtime > Restart runtime lalu lanjutkan ke cell import.")

## 2. Import Library

In [ ]:
import os
import re
import json
import shutil
import random
import warnings
from pathlib import Path
from urllib.parse import urlparse, parse_qs
from getpass import getpass
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Border, Side, Alignment
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.utils import get_column_letter
from openpyxl.formatting.rule import FormulaRule

import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 220)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Library berhasil di-import.")
print("CUDA tersedia:", torch.cuda.is_available())
print("numpy:", np.__version__)
print("pandas:", pd.__version__)

## 3. Konfigurasi Project

In [ ]:
# ============================================================
# KONFIGURASI UTAMA
# ============================================================

YOUTUBE_URLS = [
    "https://www.youtube.com/live/iAE-ltq8CFI?si=oZ-igu5wwLvglsfE"
    # Tambahkan link video lain jika komentar belum mencapai 6000.
    # "https://www.youtube.com/watch?v=VIDEO_ID_LAIN",
]

TARGET_COMMENTS = 6000
COMMENT_ORDER = "time"  # "time" atau "relevance"

RUN_SCRAPING = True
UPLOAD_MANUAL_LABEL_FILE = False

MODEL_NAME = "indolem/indobertweet-base-uncased"

MAX_LENGTH = 128
TEST_SIZE = 0.2
NUM_TRAIN_EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5

PROJECT_DIR = Path("/content/Proyek_IndoBERTweet_LCC_Kalbar_6000")
DATASET_DIR = PROJECT_DIR / "dataset"
HASIL_DIR = PROJECT_DIR / "hasil"
MODEL_DIR = PROJECT_DIR / "model"
SOURCE_CODE_DIR = PROJECT_DIR / "source_code"

for folder in [PROJECT_DIR, DATASET_DIR, HASIL_DIR, MODEL_DIR, SOURCE_CODE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_CSV = DATASET_DIR / "komentar_raw_6000.csv"
RAW_XLSX = DATASET_DIR / "komentar_raw_6000.xlsx"
LABELED_CSV = DATASET_DIR / "komentar_labeled_6000.csv"
LABELED_XLSX = DATASET_DIR / "komentar_labeled_6000_rapi.xlsx"
PREPROCESSED_CSV = DATASET_DIR / "komentar_preprocessed_6000.csv"
PREPROCESSED_XLSX = DATASET_DIR / "komentar_preprocessed_6000_rapi.xlsx"

EVALUATION_CSV = HASIL_DIR / "hasil_evaluasi_indobertweet.csv"
EVALUATION_XLSX = HASIL_DIR / "hasil_evaluasi_indobertweet.xlsx"
REPORT_TXT = HASIL_DIR / "classification_report_indobertweet.txt"

print("Project siap:", PROJECT_DIR)
print("Target komentar:", TARGET_COMMENTS)
print("Model:", MODEL_NAME)

## 4. Fungsi Styling Excel

In [ ]:
def autosize_columns(ws, min_width=10, max_width=65):
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len = 0
        for cell in col:
            if cell.value is not None:
                max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, min_width), max_width)


def style_header(ws, fill_color="1F4E78"):
    header_fill = PatternFill("solid", fgColor=fill_color)
    header_font = Font(color="FFFFFF", bold=True)
    thin = Side(border_style="thin", color="D9E2F3")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = border


def style_body(ws):
    thin = Side(border_style="thin", color="E5E7EB")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = border


def add_label_dropdown(ws, label_col_name="label"):
    headers = [cell.value for cell in ws[1]]
    if label_col_name not in headers:
        return

    col_idx = headers.index(label_col_name) + 1
    col_letter = get_column_letter(col_idx)

    dv = DataValidation(
        type="list",
        formula1='"kritik_juri_panitia,pro_peserta,netral"',
        allow_blank=False
    )
    dv.error = "Label harus salah satu dari: kritik_juri_panitia, pro_peserta, netral"
    dv.errorTitle = "Label tidak valid"
    dv.prompt = "Pilih label yang sesuai"
    dv.promptTitle = "Validasi Label"

    ws.add_data_validation(dv)
    dv.add(f"{col_letter}2:{col_letter}7000")


def add_label_conditional_format(ws, label_col_name="label"):
    headers = [cell.value for cell in ws[1]]
    if label_col_name not in headers:
        return

    col_idx = headers.index(label_col_name) + 1
    col_letter = get_column_letter(col_idx)
    max_row = ws.max_row

    color_map = {
        "kritik_juri_panitia": "F8CBAD",
        "pro_peserta": "C6E0B4",
        "netral": "D9EAF7"
    }

    for label, color in color_map.items():
        ws.conditional_formatting.add(
            f"{col_letter}2:{col_letter}{max_row}",
            FormulaRule(
                formula=[f'${col_letter}2="{label}"'],
                fill=PatternFill("solid", fgColor=color)
            )
        )


def save_dataframe_to_styled_excel(df, file_path, sheet_name="Data", label_dropdown=False):
    with pd.ExcelWriter(file_path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)

    wb = load_workbook(file_path)
    ws = wb[sheet_name]

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    ws.sheet_view.showGridLines = False

    style_header(ws)
    style_body(ws)
    autosize_columns(ws)

    headers = [cell.value for cell in ws[1]]
    width_map = {
        "comment": 62,
        "clean_comment": 62,
        "label_reason": 65,
        "hits_kritik": 50,
        "hits_pro_peserta": 50,
        "hits_netral": 45,
        "video_title": 45,
        "source_url": 45
    }
    for col_name, width in width_map.items():
        if col_name in headers:
            col_idx = headers.index(col_name) + 1
            ws.column_dimensions[get_column_letter(col_idx)].width = width

    ws.row_dimensions[1].height = 30
    for row_idx in range(2, min(ws.max_row + 1, 7000)):
        ws.row_dimensions[row_idx].height = 45

    if label_dropdown:
        add_label_dropdown(ws, "label")
        add_label_conditional_format(ws, "label")

    wb.save(file_path)

print("Fungsi Excel siap.")

## 5. Scraping 6000 Komentar YouTube

In [ ]:
def extract_video_id(youtube_url: str) -> str:
    parsed = urlparse(youtube_url)

    if parsed.hostname == "youtu.be":
        return parsed.path.strip("/")

    if parsed.hostname and "youtube.com" in parsed.hostname:
        query = parse_qs(parsed.query)
        if "v" in query:
            return query["v"][0]

        parts = parsed.path.strip("/").split("/")
        if len(parts) >= 2 and parts[0] in ["live", "shorts", "embed"]:
            return parts[1]

    raise ValueError(f"Video ID tidak ditemukan dari URL: {youtube_url}")


def get_youtube_service(api_key: str):
    return build("youtube", "v3", developerKey=api_key)


def get_video_title(youtube, video_id: str) -> str:
    try:
        request = youtube.videos().list(part="snippet", id=video_id)
        response = request.execute()
        items = response.get("items", [])
        if items:
            return items[0]["snippet"].get("title", "")
        return ""
    except Exception:
        return ""


def scrape_comments_from_video(youtube, youtube_url: str, max_comments: int, order: str = "time") -> pd.DataFrame:
    video_id = extract_video_id(youtube_url)
    video_title = get_video_title(youtube, video_id)

    comments = []
    next_page_token = None

    while len(comments) < max_comments:
        remaining = max_comments - len(comments)
        request_max = min(100, remaining)

        try:
            request = youtube.commentThreads().list(
                part="snippet,replies",
                videoId=video_id,
                maxResults=request_max,
                pageToken=next_page_token,
                textFormat="plainText",
                order=order
            )
            response = request.execute()

        except HttpError as error:
            error_text = str(error)
            print("\nError saat mengambil komentar dari video:", video_id)

            if "API key not valid" in error_text:
                print("API key tidak valid.")
            elif "commentsDisabled" in error_text:
                print("Komentar pada video ini dinonaktifkan.")
            elif "videoNotFound" in error_text:
                print("Video tidak ditemukan.")
            elif "accessNotConfigured" in error_text:
                print("YouTube Data API v3 belum diaktifkan.")
            elif "quotaExceeded" in error_text:
                print("Kuota YouTube API habis.")
            else:
                print(error)
            break

        items = response.get("items", [])
        if not items:
            break

        for item in items:
            top_comment = item["snippet"]["topLevelComment"]
            snippet = top_comment["snippet"]

            comments.append({
                "comment_id": top_comment.get("id"),
                "video_id": video_id,
                "video_title": video_title,
                "source_url": youtube_url,
                "author": snippet.get("authorDisplayName"),
                "comment": snippet.get("textDisplay"),
                "like_count": snippet.get("likeCount"),
                "published_at": snippet.get("publishedAt"),
                "updated_at": snippet.get("updatedAt"),
                "total_reply_count": item["snippet"].get("totalReplyCount", 0)
            })

            if len(comments) >= max_comments:
                break

        print(f"Video {video_id}: {len(comments)} komentar terkumpul...")

        next_page_token = response.get("nextPageToken")
        if not next_page_token:
            break

    return pd.DataFrame(comments)


def scrape_multiple_videos(youtube, urls, target_comments=6000, order="time") -> pd.DataFrame:
    all_data = []
    collected = 0

    for url in urls:
        if collected >= target_comments:
            break

        remaining = target_comments - collected
        print("\nMengambil komentar dari:", url)

        df_video = scrape_comments_from_video(
            youtube=youtube,
            youtube_url=url,
            max_comments=remaining,
            order=order
        )

        if len(df_video) > 0:
            all_data.append(df_video)
            collected += len(df_video)

        print(f"Total sementara: {collected}/{target_comments}")

    if not all_data:
        return pd.DataFrame()

    df_all = pd.concat(all_data, ignore_index=True)
    df_all = df_all.drop_duplicates(subset=["comment_id"]).reset_index(drop=True)
    df_all = df_all.drop_duplicates(subset=["comment"]).reset_index(drop=True)

    return df_all.head(target_comments)


if RUN_SCRAPING:
    YOUTUBE_API_KEY = getpass("Masukkan YouTube API Key asli dari Google Cloud: ").strip()

    if not YOUTUBE_API_KEY:
        raise ValueError("API key masih kosong.")

    youtube = get_youtube_service(YOUTUBE_API_KEY)

    df_raw = scrape_multiple_videos(
        youtube=youtube,
        urls=YOUTUBE_URLS,
        target_comments=TARGET_COMMENTS,
        order=COMMENT_ORDER
    )

    if len(df_raw) == 0:
        raise ValueError("Tidak ada komentar yang berhasil diambil.")

    if len(df_raw) < TARGET_COMMENTS:
        print(f"PERINGATAN: komentar yang berhasil diambil hanya {len(df_raw)} dari target {TARGET_COMMENTS}.")
        print("Tambahkan URL video lain yang relevan pada YOUTUBE_URLS agar mencapai 6000.")

    df_raw.to_csv(RAW_CSV, index=False, encoding="utf-8-sig")
    save_dataframe_to_styled_excel(df_raw, RAW_XLSX, sheet_name="Komentar_Raw", label_dropdown=False)

else:
    if not RAW_CSV.exists():
        raise FileNotFoundError("RUN_SCRAPING=False, tetapi komentar_raw_6000.csv belum tersedia.")
    df_raw = pd.read_csv(RAW_CSV)

print("\nDataset raw siap.")
print("Jumlah komentar:", len(df_raw))
display(df_raw.head(10))

## 6. Preprocessing Teks

In [ ]:
BASIC_STOPWORDS = {
    "yang", "dan", "di", "ke", "dari", "ini", "itu", "untuk", "dengan", "atau",
    "pada", "adalah", "karena", "sebagai", "dalam", "akan", "juga", "sudah",
    "saja", "sih", "dong", "deh", "nih", "ya", "yg", "aku", "saya", "kami",
    "kita", "mereka", "dia", "ia", "nya", "pun", "kok", "lah", "kan", "kah",
    "banget", "amat", "sangat", "lebih", "kurang"
}

try:
    from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
    sastrawi_stopwords = set(StopWordRemoverFactory().get_stop_words())
    negation_words = {"tidak", "bukan", "jangan", "belum", "tanpa", "tak"}
    STOPWORDS = BASIC_STOPWORDS.union(sastrawi_stopwords) - negation_words
    print("Stopword Sastrawi digunakan.")
except Exception:
    STOPWORDS = BASIC_STOPWORDS
    print("Sastrawi stopword tidak tersedia. Menggunakan stopword dasar.")

try:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
    stemmer = StemmerFactory().create_stemmer()
    print("Stemmer Sastrawi digunakan.")
except Exception:
    stemmer = None
    print("Stemmer tidak tersedia. Stemming dilewati.")

NORMALIZATION_DICT = {
    "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak",
    "ngga": "tidak", "tdk": "tidak", "tak": "tidak",
    "bgt": "banget", "bgtt": "banget",
    "klo": "kalau", "kl": "kalau",
    "krn": "karena", "karna": "karena",
    "dr": "dari", "dgn": "dengan",
    "sm": "sama", "sma": "sama",
    "jd": "jadi", "jdi": "jadi",
    "jg": "juga",
    "tp": "tapi", "tpi": "tapi",
    "trs": "terus", "trus": "terus",
    "utk": "untuk",
    "pd": "pada",
    "org": "orang",
    "ank": "anak",
    "adlh": "adalah",
    "hrs": "harus",
    "bs": "bisa", "bsa": "bisa",
    "knp": "kenapa",
    "smg": "semoga", "moga": "semoga",
    "mnrt": "menurut",
    "bener": "benar", "bner": "benar",
    "kasian": "kasihan", "kasiann": "kasihan",
    "jurinya": "juri",
    "pesertanya": "peserta",
    "panitianya": "panitia",
    "penilaiannya": "penilaian",
    "mprri": "mpr",
    "kalbar": "kalimantan barat"
}

def normalize_token(token):
    return NORMALIZATION_DICT.get(token, token)


def clean_basic(text):
    if pd.isna(text):
        return ""

    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text)
    text = re.sub(r"#[A-Za-z0-9_]+", " ", text)
    text = text.encode("ascii", "ignore").decode("ascii")
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def preprocess_text(text, use_stemming=True):
    text = clean_basic(text)
    tokens = text.split()
    tokens = [normalize_token(token) for token in tokens]
    tokens = [token for token in tokens if token not in STOPWORDS and len(token) > 2]

    if use_stemming and stemmer is not None:
        tokens = [stemmer.stem(token) for token in tokens]

    return " ".join(tokens)


df = df_raw.copy()
df["clean_comment"] = df["comment"].apply(preprocess_text)
df["jumlah_kata"] = df["comment"].astype(str).apply(lambda x: len(x.split()))
df["jumlah_karakter"] = df["comment"].astype(str).apply(len)
df = df[df["clean_comment"].str.strip() != ""].reset_index(drop=True)

df.to_csv(PREPROCESSED_CSV, index=False, encoding="utf-8-sig")
save_dataframe_to_styled_excel(df, PREPROCESSED_XLSX, sheet_name="Preprocessing", label_dropdown=False)

print("Jumlah data setelah preprocessing:", len(df))
display(df[["comment", "clean_comment"]].head(10))

## 7. Pelabelan Otomatis 3 Kelas

In [ ]:
def pattern_score(text, weighted_patterns):
    score, hits = 0, []
    for pattern, weight in weighted_patterns:
        if re.search(pattern, text):
            score += weight
            hits.append(pattern)
    return score, hits


def keyword_score(text, weighted_keywords):
    score, hits = 0, []
    for keyword, weight in weighted_keywords:
        if keyword in text:
            score += weight
            hits.append(keyword)
    return score, hits


KRITIK_PATTERNS = [
    (r"tidak\s+adil", 10), (r"gak\s+adil", 10), (r"ga\s+adil", 10),
    (r"nggak\s+adil", 10), (r"kurang\s+adil", 8), (r"tidak\s+fair", 9),
    (r"evaluasi\s+ulang", 9), (r"ulang\s+lomba", 9), (r"penilaian\s+ulang", 9),
    (r"juri\s+salah", 9), (r"juri\s+keliru", 9), (r"panitia\s+salah", 9),
    (r"sistem\s+kacau", 9), (r"aturan\s+aneh", 8), (r"curang", 9),
    (r"memalukan", 8), (r"kacau", 7), (r"protes", 7)
]

KRITIK_KEYWORDS = [
    ("juri", 4), ("panitia", 4), ("penilaian", 5), ("nilai", 3),
    ("dinilai", 3), ("menilai", 3), ("evaluasi", 5), ("ulang", 4),
    ("salah", 5), ("keliru", 5), ("kacau", 5), ("aneh", 4),
    ("curang", 6), ("adil", 4), ("fair", 4), ("mpr", 3),
    ("sistem", 4), ("aturan", 4), ("keputusan", 4), ("protes", 5),
    ("klarifikasi", 4), ("kontroversi", 4), ("merugikan", 6),
    ("seharusnya", 3), ("harusnya", 3), ("kenapa", 2), ("kok", 2),
    ("tanggung jawab", 5), ("jurinya", 4), ("panitianya", 4), ("nilainya", 4)
]

PRO_PATTERNS = [
    (r"kasihan\s+peserta", 10), (r"kasihan\s+anak", 10), (r"kasihan\s+adik", 10),
    (r"dukung\s+peserta", 10), (r"semangat\s+peserta", 10), (r"tetap\s+semangat", 8),
    (r"peserta\s+benar", 9), (r"siswa\s+benar", 9), (r"anak\s+itu\s+benar", 9),
    (r"peserta\s+hebat", 8), (r"adik\s+hebat", 8)
]

PRO_KEYWORDS = [
    ("peserta", 5), ("siswa", 5), ("siswi", 5), ("anak", 5), ("adik", 5),
    ("sekolah", 4), ("kasihan", 7), ("kasian", 7), ("semangat", 7),
    ("dukung", 6), ("mendukung", 6), ("support", 5), ("hebat", 5),
    ("bangga", 5), ("prestasi", 5), ("korban", 5), ("mental", 4), ("trauma", 4)
]

NETRAL_PATTERNS = [
    (r"link\s+nya", 7), (r"full\s+video", 7), (r"video\s+full", 7),
    (r"part\s+\d+", 6), (r"menit\s+berapa", 7)
]

NETRAL_KEYWORDS = [
    ("kapan", 4), ("dimana", 4), ("di mana", 4), ("siapa", 3), ("apa", 2),
    ("link", 5), ("full", 5), ("video", 3), ("nonton", 4), ("lihat", 3),
    ("info", 4), ("informasi", 4), ("hadir", 3), ("live", 3), ("menit", 4),
    ("bagian", 3), ("part", 4), ("channel", 3), ("subscribe", 5), ("like", 3), ("share", 3)
]


def classify_comment_3class(comment):
    text = clean_basic(comment)

    kritik_p, hits_kp = pattern_score(text, KRITIK_PATTERNS)
    kritik_k, hits_kk = keyword_score(text, KRITIK_KEYWORDS)
    kritik_total = kritik_p + kritik_k

    pro_p, hits_pp = pattern_score(text, PRO_PATTERNS)
    pro_k, hits_pk = keyword_score(text, PRO_KEYWORDS)
    pro_total = pro_p + pro_k

    netral_p, hits_np = pattern_score(text, NETRAL_PATTERNS)
    netral_k, hits_nk = keyword_score(text, NETRAL_KEYWORDS)
    netral_total = netral_p + netral_k

    has_authority = any(k in text for k in ["juri", "panitia", "penilaian", "nilai", "mpr", "sistem", "aturan", "keputusan"])
    has_negative = any(k in text for k in ["tidak", "gak", "ga", "nggak", "salah", "keliru", "adil", "evaluasi", "ulang", "protes", "kacau", "aneh"])

    if has_authority and has_negative:
        kritik_total += 6
        hits_kk.append("authority+negative_context")

    scores = {
        "kritik_juri_panitia": kritik_total,
        "pro_peserta": pro_total,
        "netral": netral_total
    }

    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    label, top_score = sorted_scores[0]
    second_label, second_score = sorted_scores[1]

    if top_score == 0:
        label = "netral"
        confidence = 0.50
        reason = "tidak ada indikator kuat"
    else:
        if label == "netral" and max(kritik_total, pro_total) >= 6:
            label = "kritik_juri_panitia" if kritik_total >= pro_total else "pro_peserta"
            top_score = max(kritik_total, pro_total)
            reason = "indikator sikap lebih kuat daripada indikator netral"
        else:
            reason = f"skor tertinggi: {label}={top_score}, kedua={second_label}={second_score}"

        if pro_total >= 10 and (kritik_total - pro_total) <= 5:
            label = "pro_peserta"
            reason = f"dukungan peserta dominan; pro={pro_total}, kritik={kritik_total}"

        if kritik_p >= 9 or (has_authority and has_negative and kritik_total >= pro_total + 4):
            label = "kritik_juri_panitia"
            reason = f"kritik kuat terhadap juri/panitia/sistem; kritik={kritik_total}, pro={pro_total}"

        gap = abs(top_score - second_score)
        confidence = min(0.98, 0.55 + gap * 0.03 + top_score * 0.01)

    manual_review_required = (
        confidence < 0.68 or
        (kritik_total > 0 and pro_total > 0 and abs(kritik_total - pro_total) <= 4) or
        ("wkwk" in text) or ("haha" in text) or ("mantap" in text and has_authority)
    )

    return pd.Series({
        "label": label,
        "label_confidence": round(confidence, 3),
        "label_reason": reason,
        "score_kritik": kritik_total,
        "score_pro_peserta": pro_total,
        "score_netral": netral_total,
        "hits_kritik": ", ".join((hits_kp + hits_kk)[:12]),
        "hits_pro_peserta": ", ".join((hits_pp + hits_pk)[:12]),
        "hits_netral": ", ".join((hits_np + hits_nk)[:12]),
        "manual_review_required": manual_review_required
    })


label_result = df["comment"].apply(classify_comment_3class)
df_labeled = pd.concat([df.reset_index(drop=True), label_result], axis=1)
df_labeled["label_source"] = "weighted_rule_based_3class"

df_labeled.to_csv(LABELED_CSV, index=False, encoding="utf-8-sig")
save_dataframe_to_styled_excel(df_labeled, LABELED_XLSX, sheet_name="Dataset_Label", label_dropdown=True)

print("Distribusi label:")
display(df_labeled["label"].value_counts().to_frame("jumlah"))

print("Jumlah manual review:", int(df_labeled["manual_review_required"].sum()))
display(df_labeled[["comment", "label", "label_confidence", "label_reason"]].head(15))

## 8. Opsional Upload Label Manual

In [ ]:
if UPLOAD_MANUAL_LABEL_FILE:
    from google.colab import files
    uploaded = files.upload()
    uploaded_name = list(uploaded.keys())[0]

    if uploaded_name.endswith(".xlsx"):
        df_labeled = pd.read_excel(uploaded_name)
    elif uploaded_name.endswith(".csv"):
        df_labeled = pd.read_csv(uploaded_name)
    else:
        raise ValueError("Format harus CSV atau XLSX.")

    required = {"comment", "label"}
    missing = required - set(df_labeled.columns)
    if missing:
        raise ValueError(f"Kolom wajib tidak ada: {missing}")

    df_labeled.to_csv(LABELED_CSV, index=False, encoding="utf-8-sig")
    save_dataframe_to_styled_excel(df_labeled, LABELED_XLSX, sheet_name="Dataset_Label", label_dropdown=True)
    print("Label manual berhasil digunakan.")
else:
    print("Upload label manual dilewati.")

## 9. EDA Lengkap dan Gambar Hasil

In [ ]:
label_counts = df_labeled["label"].value_counts().reset_index()
label_counts.columns = ["label", "jumlah"]
label_counts["persentase"] = (label_counts["jumlah"] / label_counts["jumlah"].sum() * 100).round(2)

display(label_counts)

label_counts.to_csv(HASIL_DIR / "ringkasan_distribusi_label.csv", index=False, encoding="utf-8-sig")
label_counts.to_excel(HASIL_DIR / "ringkasan_distribusi_label.xlsx", index=False)

plt.figure(figsize=(8, 5))
plt.bar(label_counts["label"], label_counts["jumlah"])
plt.title("Distribusi Label Sentimen Komentar YouTube")
plt.xlabel("Label")
plt.ylabel("Jumlah Komentar")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
path_bar = HASIL_DIR / "01_distribusi_label_bar.png"
plt.savefig(path_bar, dpi=150, bbox_inches="tight")
plt.show()

plt.figure(figsize=(7, 7))
plt.pie(label_counts["jumlah"], labels=label_counts["label"], autopct="%1.1f%%", startangle=90)
plt.title("Persentase Distribusi Label Sentimen")
plt.tight_layout()
path_pie = HASIL_DIR / "02_distribusi_label_pie.png"
plt.savefig(path_pie, dpi=150, bbox_inches="tight")
plt.show()

label_order = label_counts["label"].tolist()

plt.figure(figsize=(9, 5))
data_to_plot = [df_labeled[df_labeled["label"] == label]["jumlah_kata"] for label in label_order]
plt.boxplot(data_to_plot, labels=label_order)
plt.title("Sebaran Jumlah Kata Komentar per Label")
plt.xlabel("Label")
plt.ylabel("Jumlah Kata")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
path_len = HASIL_DIR / "03_panjang_komentar_per_label.png"
plt.savefig(path_len, dpi=150, bbox_inches="tight")
plt.show()

confidence_mean = df_labeled.groupby("label")["label_confidence"].mean().reset_index().sort_values("label_confidence", ascending=False)
display(confidence_mean)

plt.figure(figsize=(8, 5))
plt.bar(confidence_mean["label"], confidence_mean["label_confidence"])
plt.title("Rata-rata Confidence Pelabelan per Label")
plt.xlabel("Label")
plt.ylabel("Rata-rata Confidence")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
path_conf = HASIL_DIR / "04_rata_rata_confidence_per_label.png"
plt.savefig(path_conf, dpi=150, bbox_inches="tight")
plt.show()

review_counts = df_labeled["manual_review_required"].value_counts().reset_index()
review_counts.columns = ["manual_review_required", "jumlah"]
display(review_counts)

plt.figure(figsize=(7, 5))
plt.bar(review_counts["manual_review_required"].astype(str), review_counts["jumlah"])
plt.title("Jumlah Komentar yang Perlu Dicek Manual")
plt.xlabel("Manual Review Required")
plt.ylabel("Jumlah Komentar")
plt.tight_layout()
path_review = HASIL_DIR / "05_manual_review_required.png"
plt.savefig(path_review, dpi=150, bbox_inches="tight")
plt.show()

for label in label_order:
    text_label = " ".join(df_labeled[df_labeled["label"] == label]["clean_comment"].astype(str))
    word_counts = Counter(text_label.split()).most_common(20)

    if not word_counts:
        continue

    words_df = pd.DataFrame(word_counts, columns=["kata", "jumlah"])
    display(words_df)

    plt.figure(figsize=(10, 5))
    plt.bar(words_df["kata"], words_df["jumlah"])
    plt.title(f"Top 20 Kata pada Label: {label}")
    plt.xlabel("Kata")
    plt.ylabel("Frekuensi")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()

    safe_label = str(label).replace("/", "_").replace(" ", "_")
    path_words = HASIL_DIR / f"06_top_kata_label_{safe_label}.png"
    plt.savefig(path_words, dpi=150, bbox_inches="tight")
    plt.show()


def detect_aspect(comment):
    text = clean_basic(comment)

    aspect_keywords = {
        "juri_panitia": ["juri", "panitia", "jurinya", "panitianya"],
        "sistem_penilaian": ["penilaian", "nilai", "dinilai", "menilai", "sistem", "aturan", "keputusan", "jawaban"],
        "peserta": ["peserta", "siswa", "siswi", "anak", "adik", "sekolah"],
        "mpr_lomba": ["mpr", "lomba", "lcc", "pilar"],
        "informasi_video": ["video", "link", "full", "live", "menit", "part", "nonton"]
    }

    scores = {aspect: sum(1 for kw in kws if kw in text) for aspect, kws in aspect_keywords.items()}
    best_aspect = max(scores, key=scores.get)
    return best_aspect if scores[best_aspect] > 0 else "lainnya"


df_labeled["aspect"] = df_labeled["comment"].apply(detect_aspect)

aspect_counts = df_labeled["aspect"].value_counts().reset_index()
aspect_counts.columns = ["aspect", "jumlah"]
display(aspect_counts)

plt.figure(figsize=(9, 5))
plt.bar(aspect_counts["aspect"], aspect_counts["jumlah"])
plt.title("Distribusi Aspek Komentar YouTube")
plt.xlabel("Aspek")
plt.ylabel("Jumlah Komentar")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
path_aspect = HASIL_DIR / "07_aspect_distribution.png"
plt.savefig(path_aspect, dpi=150, bbox_inches="tight")
plt.show()

aspect_label = df_labeled.groupby(["aspect", "label"]).size().reset_index(name="jumlah")
aspect_pivot = aspect_label.pivot(index="aspect", columns="label", values="jumlah").fillna(0)

aspect_pivot.plot(kind="bar", figsize=(10, 5))
plt.title("Distribusi Label Berdasarkan Aspek")
plt.xlabel("Aspek")
plt.ylabel("Jumlah Komentar")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
path_aspect_label = HASIL_DIR / "08_aspect_vs_label.png"
plt.savefig(path_aspect_label, dpi=150, bbox_inches="tight")
plt.show()

eda_xlsx = HASIL_DIR / "ringkasan_eda_lengkap.xlsx"
with pd.ExcelWriter(eda_xlsx, engine="openpyxl") as writer:
    label_counts.to_excel(writer, index=False, sheet_name="Distribusi_Label")
    confidence_mean.to_excel(writer, index=False, sheet_name="Confidence_Label")
    review_counts.to_excel(writer, index=False, sheet_name="Manual_Review")
    aspect_counts.to_excel(writer, index=False, sheet_name="Distribusi_Aspek")
    aspect_label.to_excel(writer, index=False, sheet_name="Aspek_vs_Label")
    df_labeled.head(200).to_excel(writer, index=False, sheet_name="Contoh_Data")

print("EDA selesai. Hasil tersimpan di:", HASIL_DIR)

## 10. Split Train-Test Tanpa Scikit-Learn

In [ ]:
allowed_labels = ["kritik_juri_panitia", "pro_peserta", "netral"]

df_model = df_labeled.copy()
df_model["label"] = df_model["label"].astype(str).str.strip().str.lower()

invalid = sorted(set(df_model["label"].unique()) - set(allowed_labels))
if invalid:
    raise ValueError(f"Ada label tidak valid: {invalid}")

df_model = df_model[df_model["clean_comment"].astype(str).str.strip() != ""].reset_index(drop=True)

label2id = {label: idx for idx, label in enumerate(sorted(df_model["label"].unique()))}
id2label = {idx: label for label, idx in label2id.items()}

df_model["labels"] = df_model["label"].map(label2id)

def stratified_train_test_split(df_input, label_col="labels", test_size=0.2, seed=42):
    train_parts = []
    test_parts = []

    for _, group in df_input.groupby(label_col):
        group = group.sample(frac=1, random_state=seed).reset_index(drop=True)
        n_test = max(1, int(round(len(group) * test_size))) if len(group) > 1 else 0

        test_parts.append(group.iloc[:n_test])
        train_parts.append(group.iloc[n_test:])

    train_df = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    test_df = pd.concat(test_parts).sample(frac=1, random_state=seed).reset_index(drop=True)

    return train_df, test_df

train_split, test_split = stratified_train_test_split(df_model, "labels", TEST_SIZE, SEED)

train_df = train_split[["clean_comment", "labels"]].rename(columns={"clean_comment": "text"})
test_df = test_split[["clean_comment", "labels"]].rename(columns={"clean_comment": "text"})

print("Jumlah data training:", len(train_df))
print("Jumlah data testing :", len(test_df))

print("\nDistribusi training:")
display(train_df["labels"].map(id2label).value_counts().to_frame("jumlah"))

print("\nDistribusi testing:")
display(test_df["labels"].map(id2label).value_counts().to_frame("jumlah"))

print("Mapping label:")
print(label2id)

## 11. Tokenisasi IndoBERTweet

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_ds = train_dataset.map(tokenize_batch, batched=True)
test_ds = test_dataset.map(tokenize_batch, batched=True)

train_ds = train_ds.remove_columns(["text"])
test_ds = test_ds.remove_columns(["text"])

train_ds.set_format("torch")
test_ds.set_format("torch")

collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenisasi selesai.")
print(train_ds)
print(test_ds)

## 12. Fine-Tuning IndoBERTweet

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

def compute_metrics_manual(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    labels = np.array(labels)
    preds = np.array(preds)

    accuracy = float((labels == preds).mean())

    metrics_per_class = []
    total_support = len(labels)

    for class_id in sorted(set(labels.tolist()) | set(preds.tolist())):
        tp = int(((labels == class_id) & (preds == class_id)).sum())
        fp = int(((labels != class_id) & (preds == class_id)).sum())
        fn = int(((labels == class_id) & (preds != class_id)).sum())
        support = int((labels == class_id).sum())

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        metrics_per_class.append({
            "class_id": class_id,
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support
        })

    precision_weighted = sum(m["precision"] * m["support"] for m in metrics_per_class) / total_support
    recall_weighted = sum(m["recall"] * m["support"] for m in metrics_per_class) / total_support
    f1_weighted = sum(m["f1"] * m["support"] for m in metrics_per_class) / total_support

    return {
        "accuracy": accuracy,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted
    }

try:
    training_args = TrainingArguments(
        output_dir=str(MODEL_DIR / "indobertweet_checkpoints"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        logging_steps=50,
        report_to="none",
        seed=SEED
    )
except TypeError:
    training_args = TrainingArguments(
        output_dir=str(MODEL_DIR / "indobertweet_checkpoints"),
        evaluation_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1_weighted",
        logging_steps=50,
        report_to="none",
        seed=SEED
    )

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    data_collator=collator,
    compute_metrics=compute_metrics_manual
)

trainer.train()

metrics = trainer.evaluate()
print("Hasil evaluasi:")
print(metrics)

## 13. Evaluasi Model Tanpa Scikit-Learn

In [ ]:
pred_output = trainer.predict(test_ds)
logits = pred_output.predictions
y_pred = np.argmax(logits, axis=1)
y_true = np.array(test_df["labels"])

y_pred_label = [id2label[int(i)] for i in y_pred]
y_true_label = [id2label[int(i)] for i in y_true]

eval_metrics = compute_metrics_manual((logits, y_true))
result_df = pd.DataFrame([eval_metrics])

result_df.to_csv(EVALUATION_CSV, index=False, encoding="utf-8-sig")
result_df.to_excel(EVALUATION_XLSX, index=False)

print("Hasil metrik:")
display(result_df)

def make_classification_report(y_true, y_pred, labels_order):
    rows = []

    for label_name in labels_order:
        tp = sum((yt == label_name and yp == label_name) for yt, yp in zip(y_true, y_pred))
        fp = sum((yt != label_name and yp == label_name) for yt, yp in zip(y_true, y_pred))
        fn = sum((yt == label_name and yp != label_name) for yt, yp in zip(y_true, y_pred))
        support = sum(yt == label_name for yt in y_true)

        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

        rows.append({
            "label": label_name,
            "precision": precision,
            "recall": recall,
            "f1_score": f1,
            "support": support
        })

    return pd.DataFrame(rows)

labels_sorted = sorted(df_model["label"].unique())

report_df = make_classification_report(y_true_label, y_pred_label, labels_sorted)
report_df.to_csv(HASIL_DIR / "classification_report_indobertweet.csv", index=False, encoding="utf-8-sig")
report_df.to_excel(HASIL_DIR / "classification_report_indobertweet.xlsx", index=False)

with open(REPORT_TXT, "w", encoding="utf-8") as f:
    f.write(report_df.to_string(index=False))

print("Classification report:")
display(report_df)

# Confusion matrix manual
cm = pd.DataFrame(0, index=labels_sorted, columns=labels_sorted)

for actual, pred in zip(y_true_label, y_pred_label):
    cm.loc[actual, pred] += 1

display(cm)

plt.figure(figsize=(7, 6))
plt.imshow(cm.values)
plt.title("Confusion Matrix - IndoBERTweet")
plt.xticks(range(len(labels_sorted)), labels_sorted, rotation=25, ha="right")
plt.yticks(range(len(labels_sorted)), labels_sorted)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

for i in range(len(labels_sorted)):
    for j in range(len(labels_sorted)):
        plt.text(j, i, str(cm.values[i, j]), ha="center", va="center")

plt.tight_layout()
cm_png = HASIL_DIR / "confusion_matrix_indobertweet.png"
plt.savefig(cm_png, dpi=150, bbox_inches="tight")
plt.show()

metric_plot_df = result_df.T.reset_index()
metric_plot_df.columns = ["metric", "score"]

plt.figure(figsize=(8, 5))
plt.bar(metric_plot_df["metric"], metric_plot_df["score"])
plt.title("Hasil Evaluasi Fine-Tuning IndoBERTweet")
plt.xlabel("Metrik")
plt.ylabel("Skor")
plt.ylim(0, 1)
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
metric_png = HASIL_DIR / "hasil_metrik_indobertweet.png"
plt.savefig(metric_png, dpi=150, bbox_inches="tight")
plt.show()

hasil_prediksi = pd.DataFrame({
    "label_aktual": y_true_label,
    "label_prediksi": y_pred_label
})
hasil_prediksi.to_csv(HASIL_DIR / "hasil_prediksi_testset.csv", index=False, encoding="utf-8-sig")
hasil_prediksi.to_excel(HASIL_DIR / "hasil_prediksi_testset.xlsx", index=False)

compare_df = pd.DataFrame({
    "Aktual": pd.Series(y_true_label).value_counts(),
    "Prediksi": pd.Series(y_pred_label).value_counts()
}).fillna(0)

compare_df.plot(kind="bar", figsize=(8, 5))
plt.title("Perbandingan Distribusi Label Aktual dan Prediksi")
plt.xlabel("Label")
plt.ylabel("Jumlah")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
pred_compare_png = HASIL_DIR / "prediksi_vs_label_aktual.png"
plt.savefig(pred_compare_png, dpi=150, bbox_inches="tight")
plt.show()

print("File hasil evaluasi tersimpan di:", HASIL_DIR)

## 14. Simpan Model

In [ ]:
FINAL_MODEL_DIR = MODEL_DIR / "indobertweet_finetuned"
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

with open(FINAL_MODEL_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, ensure_ascii=False, indent=2)

print("Model berhasil disimpan di:", FINAL_MODEL_DIR)

## 15. Prediksi Komentar Baru

In [ ]:
def predict_sentiment(texts):
    if isinstance(texts, str):
        texts = [texts]

    encoded = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    encoded = {k: v.to(device) for k, v in encoded.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**encoded)
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1).cpu().numpy()

    results = []
    for text, pred, prob in zip(texts, preds, probs.cpu().numpy()):
        results.append({
            "komentar": text,
            "prediksi_label": id2label[int(pred)],
            "confidence": round(float(np.max(prob)), 4)
        })

    return pd.DataFrame(results)


sample_comments = [
    "Juri dan panitia harus evaluasi ulang sistem penilaiannya.",
    "Kasihan pesertanya, semoga tetap semangat.",
    "Ini full videonya ada dimana?",
    "Keputusan lomba ini menurut saya kurang adil."
]

display(predict_sentiment(sample_comments))

## 16. Buat ZIP Project

In [ ]:
ZIP_PATH = "/content/Proyek_IndoBERTweet_LCC_Kalbar_6000"

zip_file = Path(f"{ZIP_PATH}.zip")
if zip_file.exists():
    zip_file.unlink()

shutil.make_archive(ZIP_PATH, "zip", PROJECT_DIR)

print("ZIP berhasil dibuat:")
print(f"{ZIP_PATH}.zip")

try:
    from google.colab import files
    files.download(f"{ZIP_PATH}.zip")
except Exception:
    print("Download otomatis hanya tersedia di Google Colab.")

## 17. Catatan Akhir

Notebook ini adalah versi terbaru yang dibuat tanpa `scikit-learn`, sehingga lebih aman dari error environment Colab.

Yang perlu diperhatikan:

1. Jika data tidak mencapai 6000 komentar, tambahkan URL video lain yang relevan.
2. Label otomatis adalah label awal. Komentar yang `manual_review_required=True` sebaiknya dicek manual.
3. Jalankan menggunakan GPU agar fine-tuning IndoBERTweet lebih cepat.
4. Semua file hasil akan tersimpan otomatis pada folder project dan bisa diunduh sebagai ZIP.
